# 19 — Knowledge Distillation and Synthetic Data

**Network LLM Engineering — Part IV — Post-Training**

### Learning goals
- Understand teacher/student distillation
- Design a network synthetic-data pipeline
- Use validators to stop synthetic errors from becoming training truth

## Distillation

A stronger **teacher** provides signals that train a smaller **student**.
Signals can be:
- teacher-generated answers,
- preference rankings,
- logits/probabilities when accessible,
- intermediate labels/structured rationales,
- verified task trajectories.

The business motivation is often local deployment, lower latency, or lower serving cost.

In [ ]:
# Temperature-softened probability demo for logits distillation.
import torch
teacher_logits = torch.tensor([5.0, 2.0, 0.2])
for T in [1.0, 2.0, 4.0]:
    probs = torch.softmax(teacher_logits/T, dim=-1)
    print("T=",T, "teacher target distribution:", probs.tolist())

## A safe network synthetic-data flywheel

`source incident -> sanitize -> generate variants -> deterministic/domain validation -> expert sample review -> train candidate -> held-out eval`

Never assume "generated by a strong model" means correct.
Synthetic errors can be amplified during fine-tuning.

In [ ]:
import ipaddress, random, json

def synthetic_subnet_case():
    prefix = random.choice([24,25,26,27,28])
    net = ipaddress.ip_network(f"192.0.2.0/{prefix}", strict=False)
    return {
        "prompt": f"How many addresses are in a /{prefix} IPv4 subnet?",
        "answer": net.num_addresses,
        "verifier": "python ipaddress"
    }

for _ in range(3):
    print(synthetic_subnet_case())

## Distill behavior, not stale state

Teacher-generated examples should emphasize stable network reasoning patterns.
Current production state should still be retrieved at runtime.

### Exercise

Design a teacher -> validator -> student pipeline for BGP troubleshooting.
What parts can be verified automatically, and what still needs expert review?